# ngVLA Band 5 simulation: Gaussian disks and point sources

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/ngvla_band5_gaussian_disk_simulation.ipynb)

Simulate a five-channel ngVLA Band 5 (30.5 to 50.5 GHz) observation of a **protoplanetary disk built
from nested Gaussian rings** ("Gaussian disks") and two field point sources with the ngVLA core
subarray, verify the visibilities against the analytic model **including the w term**, image the five
channels and check the recovered fluxes.

---
## The Gaussian-ring component

A Gaussian-broadened ring is an infinitesimally thin circular ring of radius $R$ convolved with a circular
Gaussian of width $\sigma$ (FWHM $2\sqrt{2\ln 2}\,\sigma$), inclined by $i$ and rotated to position angle
$\psi$.  A ring of radius zero is a circular Gaussian; nested rings with independent radii, widths, fluxes
and even orientations model a protoplanetary disk.  Face on and in the tangent plane its visibility is

$$ V(u, v) = F\, J_0(2\pi R \rho)\, \exp(-2\pi^2\sigma^2\rho^2), $$

with $\rho$ the deprojected baseline length when inclined.  The simulator evaluates the full form of the
memo *Analytic visibility models for the simulator sky components, including the w term* (version 2):
the baseline coordinates are rotated into the frame of each component and the quadratic ($w$) phase
across the component is kept, which turns $\sigma^2$ into a complex width and adds a Neumann series in
the inclination.  For this compact-array, high-frequency case the $w$ term is tiny (a cell below
quantifies it); for the VLA and the ngVLA main array at low frequencies it is not.

The rings enter through ``gaussian_ring_source_flux``, ``gaussian_ring_source_ra_dec`` and
``gaussian_ring_source_shape = [radius, fwhm, inclination, position_angle]``.

---
## API

In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
# Set True for interactive (zoom / pan) plots via the ipympl widget backend
# (``pip install ipympl``).  Keep False for the automated notebook tests, which
# execute headless.
INTERACTIVE_PLOTS = False

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from astropy.coordinates import SkyCoord
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_PLOTS else "inline")

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)
SPEED_OF_LIGHT = 299792458.0

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="6GB")
viper_client

## The ngVLA core subarray and five Band 5 channels

The revision D core: 114 antennas of 18 m within 4.3 km.  Five 1 GHz channels spaced 5 GHz apart cover
Band 5 (30.5 to 50.5 GHz); across the band the resolution improves by a factor 1.7 and the primary beam
shrinks by the same factor.

In [ ]:
from astroviper.utils.telescope_layout import read_telescope_layout

antenna_xds = read_telescope_layout("ngvla-revD.core")
n_antenna = antenna_xds.sizes["antenna_name"]
positions = antenna_xds.ANTENNA_POSITION.values
baselines = np.linalg.norm(positions[:, None, :] - positions[None, :, :], axis=-1)
baselines = baselines[np.triu_indices(n_antenna, 1)]
dish_diameter = float(antenna_xds.ANTENNA_DISH_DIAMETER.values[0])

frequency_params = {
    "freq_start": 30.5e9,
    "freq_delta": 5e9,
    "n_channels": 5,
    "channel_width": 1e9,
    "spectral_window_name": "Band5",
}
frequencies = frequency_params["freq_start"] + frequency_params[
    "freq_delta"
] * np.arange(5)
wavelengths = SPEED_OF_LIGHT / frequencies
print(
    f"{n_antenna} antennas, {len(baselines)} baselines from {baselines.min():.0f} to {baselines.max():.0f} m"
)
pd.DataFrame(
    {
        "frequency [GHz]": frequencies / 1e9,
        "resolution lambda/b_max [arcsec]": wavelengths
        / baselines.max()
        / ARCSEC_TO_RAD,
        "primary beam FWHM 1.02 lambda/D [arcsec]": 1.02
        * wavelengths
        / dish_diameter
        / ARCSEC_TO_RAD,
        "longest baseline [k-lambda]": baselines.max() / wavelengths / 1e3,
    }
).round(2)

## The sky: a Gaussian disk and two field point sources

Three nested rings share the inclination (40 degrees) and position angle (60 degrees) of an HL Tau-like
disk at 140 pc: radii 0.6'', 1.3'' and 2.0'' (84, 182 and 280 au) with FWHM widths of 0.15'' to 0.25'',
flat spectra so the flux checks are exact, and no emission at the centre, so the ring structure is
unmistakable.  Two field point sources are placed by image pixel 12'' and 16'' from the phase centre,
where the primary beam varies noticeably across the band.

In [ ]:
from astroviper.utils.coordinate_transforms import sin_pixel_to_celestial_coord

image_size = np.array([768, 768])
cell_size_arcsec = 0.05
cell_size = np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD

phase_center = SkyCoord(ra="04h31m38.4s", dec="+18d13m57.7s", frame="icrs")
phase_center_ra_dec = np.array([phase_center.ra.rad, phase_center.dec.rad])[None, :]

inclination = np.deg2rad(40.0)
position_angle = np.deg2rad(60.0)
ring_radius_arcsec = np.array([0.6, 1.3, 2.0])
ring_fwhm_arcsec = np.array([0.15, 0.20, 0.25])
ring_flux_mjy = np.array([3.0, 4.0, 6.0])

gaussian_ring_source_shape = np.stack(
    [
        ring_radius_arcsec * ARCSEC_TO_RAD,
        ring_fwhm_arcsec * ARCSEC_TO_RAD,
        np.full(3, inclination),
        np.full(3, position_angle),
    ],
    axis=1,
)
gaussian_ring_source_flux = np.stack(
    [np.array([f, 0, 0, f]) * 1e-3 for f in ring_flux_mjy]
)[:, None, None, :]
gaussian_ring_source_ra_dec = np.repeat(phase_center_ra_dec[None, :, :], 3, axis=1)

pixels_point = np.array([[584, 264], [124, 564]])  # source A, source B
flux_point_mjy = np.array([3.0, 2.0])
point_source_ra_dec = sin_pixel_to_celestial_coord(
    phase_center_ra_dec[0], image_size, cell_size, pixels_point
)[None, :, :]
point_source_flux = np.stack([np.array([f, 0, 0, f]) * 1e-3 for f in flux_point_mjy])[
    :, None, None, :
]
offsets_arcsec = np.hypot(*((pixels_point - image_size // 2) * cell_size_arcsec).T)
print("point-source offsets from the phase centre [arcsec]:", offsets_arcsec.round(1))
print(
    f"total flux: rings {ring_flux_mjy.sum():.1f} mJy, points {flux_point_mjy.sum():.1f} mJy"
)

## Simulate

Two hours around transit in 600 s integrations, `XX` and `YY`, no thermal noise (see the noise
tutorial for `noise_params`).

In [ ]:
from astroviper.utils.beam_models import airy_disk_model

ngvla_beam = airy_disk_model("ngvla")
result = simulate_processing_set(
    ps_store="ngvla_band5.ps.zarr",
    antenna_xds=antenna_xds,
    time_params={
        "time_start": "2021-10-03T09:55:00.000",
        "time_delta": 600.0,
        "n_samples": 12,
    },
    frequency_params=frequency_params,
    polarization=["XX", "YY"],
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    gaussian_ring_source_flux=gaussian_ring_source_flux,
    gaussian_ring_source_ra_dec=gaussian_ring_source_ra_dec,
    gaussian_ring_source_shape=gaussian_ring_source_shape,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=[ngvla_beam],
    beam_model_map=np.zeros(n_antenna, dtype=int),
    n_time_chunks=3,
    n_frequency_chunks=5,
    overwrite=True,
)
result["timing_node_tasks"][
    ["task_id", "n_time", "n_frequency", "T_uvw", "T_visibilities", "T_write"]
]

## Visibilities against the analytic model, and the size of the w term

The measurement equation of this field is the sum of the three analytic ring responses at the phase
centre, $\sum_j F_j\,T_j(u, v, w)$, and of the two point sources, each with its exact phase
$2\pi[u l + v m + w(n - 1)]$, the $1/n$ of the simulator's flux convention and the 18 m primary
beam at its position.  The simulated visibilities must reproduce that sum to rounding.  Evaluating the
ring responses with $w = 0$ shows how large the $w$ term is here: with baselines of at most 4.3 km at
30 to 50 GHz and rings of at most 2'', the Fresnel parameter $\pi w R^2$ is of order $10^{-4}$, so the
tangent-plane approximation would have been fine.  The last line evaluates the outer ring on a 1000 km
ngVLA main-array baseline at Band 1 (2.4 GHz), where the parameter is a hundred times larger.

In [ ]:
from xradio.measurement_set import open_processing_set

from astroviper.processing_functions.imaging.primary_beam.airy_disk import (
    casa_airy_disk_response,
)
from astroviper.processing_functions.simulation import gaussian_ring_uv_response
from astroviper.utils.coordinate_transforms import sin_project

ps_xdt = open_processing_set("ngvla_band5.ps.zarr")
ms_xds = ps_xdt[result["ms_name"]].ds
uvw = ms_xds.UVW.values  # [time, baseline, 3] metres
visibility = ms_xds.VISIBILITY.sel(
    polarization="XX"
).values  # [time, baseline, frequency]
u = uvw[:, :, 0, None] * ms_xds.frequency.values / SPEED_OF_LIGHT
v = uvw[:, :, 1, None] * ms_xds.frequency.values / SPEED_OF_LIGHT
w = uvw[:, :, 2, None] * ms_xds.frequency.values / SPEED_OF_LIGHT


def analytic_rings(u, v, w):
    """Analytic visibility of the three rings at the phase centre [Jy]."""
    total = np.zeros(np.broadcast(u, v, w).shape, dtype=complex)
    for flux, shape in zip(ring_flux_mjy, gaussian_ring_source_shape, strict=True):
        total += flux * 1e-3 * gaussian_ring_uv_response(u, v, *shape, w=w)
    return total


# the point sources: exact phase and 1/n (the simulator's flux convention) times the
# 18 m power beam at their positions in every channel
analytic_points = np.zeros_like(visibility)
primary_beam_at_sources = np.empty((5, len(pixels_point)))
for i_source, ra_dec in enumerate(point_source_ra_dec[0]):
    l_source, m_source = sin_project(phase_center_ra_dec[0], ra_dec)
    n_source = np.sqrt(1 - l_source**2 - m_source**2)
    primary_beam_at_sources[:, i_source] = casa_airy_disk_response(
        np.array([l_source]),
        np.array([m_source]),
        frequencies,
        ngvla_beam["dish_diameter"],
        ngvla_beam["blockage_diameter"],
        ngvla_beam["max_rad_1GHz"],
        ipower=2,
    )
    analytic_points += (
        flux_point_mjy[i_source]
        * 1e-3
        * primary_beam_at_sources[None, None, :, i_source]
        * np.exp(2j * np.pi * (u * l_source + v * m_source + w * (n_source - 1)))
        / n_source
    )

expected = analytic_rings(u, v, w) + analytic_points
print(
    f"max |simulated - analytic (with w)|           = {np.abs(visibility - expected).max():.2e} Jy"
)
print(
    f"max |rings with w - rings in the tangent plane| = {np.abs(analytic_rings(u, v, w) - analytic_rings(u, v, 0.0)).max():.2e} Jy"
)
outer_radius = ring_radius_arcsec.max() * ARCSEC_TO_RAD
print(
    f"largest Fresnel parameter pi w R^2 here: {np.pi * np.abs(w).max() * outer_radius**2:.1e}  (|w| up to {np.abs(w).max() / 1e3:.0f} k-lambda)"
)
w_main_band1 = 1000e3 / (SPEED_OF_LIGHT / 2.4e9)
print(
    f"the outer ring on a 1000 km baseline at 2.4 GHz: pi w R^2 = {np.pi * w_main_band1 * outer_radius**2:.1e}"
)

The figure plots the amplitude of the disk alone (the analytic point-source terms removed from the
simulated visibilities) against the deprojected baseline length in wavelengths.  Because the rings are
circular in the deprojected frame, every visibility of every channel falls on one curve, the analytic
response along the disk's major axis: a higher frequency simply samples the same curve at larger
spatial frequencies.  The oscillations are the Bessel functions of the three ring radii.

In [ ]:
q = np.cos(inclination)
u_major = u * np.sin(position_angle) + v * np.cos(position_angle)
u_minor = u * np.cos(position_angle) - v * np.sin(position_angle)
deprojected = np.sqrt(u_major**2 + (q * u_minor) ** 2)
disk_visibility = visibility - analytic_points
curve = np.linspace(0, deprojected.max(), 1500)
analytic_curve = analytic_rings(
    curve * np.sin(position_angle), curve * np.cos(position_angle), 0.0
)
fig, ax = plt.subplots(figsize=(9, 4.2), constrained_layout=True)
colors = plt.cm.viridis(np.linspace(0, 0.9, 5))
for channel in range(5):
    ax.plot(
        deprojected[..., channel].ravel() / 1e3,
        np.abs(disk_visibility[..., channel]).ravel() * 1e3,
        ".",
        ms=2,
        alpha=0.25,
        color=colors[channel],
        label=f"{frequencies[channel] / 1e9:.1f} GHz",
    )
ax.plot(
    curve / 1e3,
    np.abs(analytic_curve) * 1e3,
    "k-",
    lw=1,
    label="analytic, along the major axis",
)
ax.set_xlabel("deprojected baseline [k$\\lambda$]")
ax.set_ylabel("|V| of the disk [mJy]")
ax.set_title("Gaussian disk: three nested rings")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, markerscale=4)
plt.show()

## Image the five channels

One node task per channel (``n_mapping_parallelism={"frequency": 5}``), Briggs weighting, a 0.05''
cell (six pixels per beam at 50.5 GHz) and Hogbom CLEAN.  No primary-beam correction is requested:
the imager's built-in primary beam is CASA's Airy pattern of the ALMA 12 m antenna, which does not
describe the 18 m ngVLA dishes, so the checks below divide the restored peaks by the simulator's own
18 m Airy beam instead (the same ``casa_airy_disk_response`` the simulator applied).

In [ ]:
from xradio.image import load_image, make_empty_sky_image

from astroviper.distributed_applications.imaging import image_cube_single_field


def image_simulation(
    ps_store, image_store, niter=4000, deconvolver="hogbom_many_threads"
):
    """Image a simulated processing set with AstroVIPER; returns the image and the imager's result dict."""
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    result = image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params={
            "image_size": list(image_size),
            "cell_size": cell_size,
            "phase_direction": phase_direction,
            "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
            "polarization_coords": ["I"],
            "time_coords": [0],
            "fft_padding": 1.2,
            "cpp_gridder": True,
        },
        imaging_weights_params={
            "weighting": "briggs",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params={
            "niter": niter,
            "nmajor": -1,
            "threshold": 5e-6,
            "gain": 0.1,
            "cyclefactor": 1.5,
            "cycleniter": -1,
            "minpsffraction": 0.05,
            "maxpsffraction": 0.8,
            "primary_beam_limit": 0.2,
        },
        gridder="prolate_spheroidal",
        deconvolver=deconvolver,
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=[
            "sky_residual",
            "point_spread_function",
            "beam_fit_params_point_spread_function",
            "sky_model",
            "mask",
        ],
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_mapping_parallelism={"frequency": 5},
        overwrite=True,
        restore=True,
        primary_beam_correction=False,
    )
    return add_sky_coordinates(load_image(image_store)), result


def add_sky_coordinates(img_xds):
    """Attach 2-D right_ascension / declination pixel coordinates (SIN grid)."""
    img_xds.attrs["type"] = "image_dataset"  # the xr_img accessor checks this
    reference_direction = img_xds.attrs["coordinate_system_info"][
        "reference_direction"
    ]["data"]
    template = make_empty_sky_image(
        phase_center=np.asarray(reference_direction),
        image_size=[img_xds.sizes["l"], img_xds.sizes["m"]],
        cell_size=img_xds.xr_img.get_lm_cell_size(),
        frequency_coords=img_xds.frequency.values,
        pol_coords=list(img_xds.polarization.values),
        time_coords=list(img_xds.time.values),
        do_sky_coords=True,
    )
    return img_xds.assign_coords(
        right_ascension=(("l", "m"), template.right_ascension.values),
        declination=(("l", "m"), template.declination.values),
    )


clean, imaging_result = image_simulation("ngvla_band5.ps.zarr", "ngvla_band5.img.zarr")
beam_fits = clean.BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION.isel(
    time=0, polarization=0
).values  # [frequency, 3]
pd.DataFrame(
    {
        "frequency [GHz]": frequencies / 1e9,
        "beam major [arcsec]": beam_fits[:, 0] / ARCSEC_TO_RAD,
        "beam minor [arcsec]": beam_fits[:, 1] / ARCSEC_TO_RAD,
        "beam pa [deg]": np.rad2deg(beam_fits[:, 2]),
    }
).round(3)

## The restored disk across the band, against the model

Top row: the restored disk in each channel (a 6'' field).  Bottom row: the difference from the
expected image, the input rings (``gaussian_ring_image``) convolved with the fitted clean beam of that
channel using the restore step's own kernel; the two field point sources enter the expected image
attenuated by the 18 m primary beam at their positions.  The rings, 0.7'' apart, blend into a broad
ellipse at 30.5 GHz (0.6'' beam) and separate as the beam shrinks to 0.4'' at 50.5 GHz.

In [ ]:
import scipy.fft

from astroviper.processing_functions.imaging.restore import _elliptical_gaussian_kernel
from astroviper.processing_functions.simulation import gaussian_ring_image

l_grid, m_grid = np.meshgrid(clean.l.values, clean.m.values, indexing="ij")
pixel_area = float(np.abs(cell_size[0] * cell_size[1]))
pixel = cell_size_arcsec * ARCSEC_TO_RAD
rings = np.zeros(tuple(image_size))
for flux, shape in zip(ring_flux_mjy, gaussian_ring_source_shape, strict=True):
    rings += flux * 1e-3 * gaussian_ring_image(l_grid, m_grid, *shape) * pixel_area

restored = clean.SKY_RESTORED.isel(time=0, polarization=0).values  # [frequency, l, m]
expected_restored = np.empty_like(restored)
for channel in range(5):
    sky = rings.copy()  # the disk sits at the phase centre, where the primary beam is 1
    for i_source, pix in enumerate(pixels_point):
        sky[tuple(pix)] += (
            flux_point_mjy[i_source] * 1e-3 * primary_beam_at_sources[channel, i_source]
        )
    kernel = _elliptical_gaussian_kernel(
        *image_size,
        beam_fits[channel, 0] / pixel,
        beam_fits[channel, 1] / pixel,
        beam_fits[channel, 2],
        np.float64,
    )
    expected_restored[channel] = scipy.fft.irfft2(
        scipy.fft.rfft2(sky) * scipy.fft.rfft2(scipy.fft.ifftshift(kernel)),
        s=tuple(image_size),
    )
difference = restored - expected_restored

centre = image_size // 2
half = 60  # 3 arcsec
zoom = (
    slice(centre[0] - half, centre[0] + half),
    slice(centre[1] - half, centre[1] + half),
)
extent = np.array([half, -half, -half, half]) * cell_size_arcsec
fig, axes = plt.subplots(2, 5, figsize=(17, 7), constrained_layout=True)
for channel in range(5):
    vmax = np.nanmax(expected_restored[channel][zoom]) * 1e3
    im = axes[0, channel].imshow(
        restored[channel][zoom].T * 1e3,
        origin="lower",
        extent=extent,
        cmap="viridis",
        vmin=0,
        vmax=vmax,
    )
    axes[0, channel].set_title(
        f"{frequencies[channel] / 1e9:.1f} GHz, restored", fontsize=10
    )
    fig.colorbar(im, ax=axes[0, channel], shrink=0.8, label="mJy/beam")
    diff = difference[channel][zoom].T * 1e3
    im = axes[1, channel].imshow(
        diff,
        origin="lower",
        extent=extent,
        cmap="RdBu_r",
        vmin=-np.abs(diff).max(),
        vmax=np.abs(diff).max(),
    )
    axes[1, channel].set_title("restored - expected", fontsize=10)
    fig.colorbar(im, ax=axes[1, channel], shrink=0.8, label="mJy/beam")
for ax in axes.ravel():
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
plt.show()

# the whole field in the last channel: the disk and the two field point sources
fig, ax = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
full_extent = (
    np.array(
        [
            image_size[0] // 2,
            -image_size[0] // 2,
            -image_size[1] // 2,
            image_size[1] // 2,
        ]
    )
    * cell_size_arcsec
)
im = ax.imshow(
    np.sqrt(np.clip(restored[-1].T * 1e3, 0, None)),
    origin="lower",
    extent=full_extent,
    cmap="viridis",
)
ax.set_title(
    f"{frequencies[-1] / 1e9:.1f} GHz, sqrt(restored) over the full 38'' field"
)
ax.set_xlabel("l [arcsec]")
ax.set_ylabel("m [arcsec]")
fig.colorbar(im, ax=ax, label="sqrt(mJy/beam)")
plt.show()

## Flux recovery per channel

The disk flux is the sum of the restored image within 3'' of the centre (times pixel area over beam
area); it must equal the 13 mJy of the rings in every channel.  For the field point sources the table
lists the 18 m primary beam at their positions, which drops across the band, and the restored peak
divided by it, which must stay at the input flux.

In [ ]:
rows = []
for channel in range(5):
    beam_area = np.pi / (4 * np.log(2)) * beam_fits[channel, 0] * beam_fits[channel, 1]
    row = {
        "frequency [GHz]": frequencies[channel] / 1e9,
        "disk flux [mJy]": np.nansum(restored[channel][zoom])
        * pixel_area
        / beam_area
        * 1e3,
        "disk max |restored - expected| / peak": np.nanmax(
            np.abs(difference[channel][zoom])
        )
        / np.nanmax(expected_restored[channel][zoom]),
    }
    for name, i_source in zip(["source A", "source B"], [0, 1], strict=True):
        pb = primary_beam_at_sources[channel, i_source]
        row[f"{name} primary beam"] = pb
        row[f"{name} peak / primary beam [mJy]"] = (
            restored[channel][tuple(pixels_point[i_source])] / pb * 1e3
        )
    rows.append(row)
table = pd.DataFrame(rows).round(4)
print(
    f"expected: disk {ring_flux_mjy.sum():.1f} mJy, "
    f"source A {flux_point_mjy[0]:.1f} mJy, source B {flux_point_mjy[1]:.1f} mJy"
)
table

In [ ]:
# flux and shape checks in every channel
np.testing.assert_allclose(table["disk flux [mJy]"], ring_flux_mjy.sum(), rtol=0.03)
assert (table["disk max |restored - expected| / peak"] < 0.03).all()
np.testing.assert_allclose(
    table["source A peak / primary beam [mJy]"], flux_point_mjy[0], rtol=0.03
)
np.testing.assert_allclose(
    table["source B peak / primary beam [mJy]"], flux_point_mjy[1], rtol=0.03
)
assert (
    table["source B primary beam"].iloc[0]
    > table["source B primary beam"].iloc[-1] + 0.05
)
print("fluxes recovered in all five channels")

## Clean up

In [ ]:
import shutil

for path in ["ngvla_band5.ps.zarr", "ngvla_band5.img.zarr"]:
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()